# analysis — 분절 그림과 성능 지표

파이프라인 스크립트(`02`~`07`)의 함수를 **그대로 불러 쓴다.** 계산 로직을 여기서 다시
쓰지 않는다 — 값이 갈라지면 어느 쪽이 맞는지 알 수 없기 때문이다.

| 절 | 내용 |
|---|---|
| 1 | 준비 — 모델·데이터 올리기 |
| 2 | 분절 고르기 — 이름·기록·SNR·복원 순위로 |
| 3 | 그림 — 고른 분절 하나에 대해 5종 |
| 4 | 성능 지표 — 03·04·06 직접 계산 |
| 5 | 저장된 산출물 읽기 — 재계산 없이 |


## 1. 준비


In [ ]:
import importlib.util, os, sys
import numpy as np, pandas as pd, torch

sys.path.insert(0, '.')

from src import metrics
from src.core import (aggregate, component_bank, enc_names, load_ckpt,
                      mad_matrix, pearson, rmse_norm_matrix)
from src.data.build import load_cfg
from src.data.dataset import REF_KEYS, load as load_split
from src.model import meae
from src.viz import plt


def script(name):
    """번호로 시작하는 파일명은 import 문으로 못 부른다. 경로로 올린다."""
    spec = importlib.util.spec_from_file_location(name[:-3], name)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


S03 = script('03_bss.py')
S04 = script('04_masked_denoising.py')
S05 = script('05_figure.py')
S06 = script('06_ablation.py')
pd.set_option('display.width', 260)
print('준비 완료')


In [ ]:
# ---- 여기만 바꾸면 된다 -------------------------------------------------
RUN   = 'C16_seed42'        # <ROOT>/02_model/<RUN>/<이름>.pt
SPLIT = 'test'              # 'val' | 'test'
ROOT  = 'results'           # 탐색 런이면 'experiments'
# -------------------------------------------------------------------------

cfg = load_cfg('configs/default.yaml')
fs = cfg['data']['fs']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = os.path.join(ROOT, '02_model', RUN, os.path.basename(RUN) + '.pt')

model, ck = load_ckpt(cfg, ckpt)
model = model.to(dev).eval()
ds = load_split(cfg, SPLIT)
pad, K = model.pad_each, model.n_encoders
sup = list(cfg['loss']['supervise'])
k_clean = sup.index('x_clean')
k_noise = [k for k in range(K) if k != k_clean]

clean = ds.refs['x_clean'].astype(np.float64)
noisy = ds.x_noisy.astype(np.float64)
snr_in = metrics.snr_db_vec(clean, noisy)
names = ['{}_{:04d}'.format(m['record_id'], m['seg_idx']) for m in ds.meta]

print('{} 에폭 {} · {} {}분절 · K={} · 압축률 {}'.format(
    RUN, ck['epoch'], SPLIT, len(ds), K, getattr(model, 'downsample', 256)))
print('배정 ' + ' · '.join('enc{}={}'.format(i + 1, r) for i, r in enumerate(sup)))
print('입력 SNR  최소 {:.2f} · 중앙 {:.2f} · 최대 {:.2f} dB'.format(
    snr_in.min(), np.median(snr_in), snr_in.max()))


## 2. 분절 고르기

이름으로 직접 찾거나, 기록·SNR 로 거르거나, 복원 순위로 뽑는다.


In [ ]:
@torch.no_grad()
def infer(i):
    """분절 하나의 성분·재구성·복원. 모델이 정의한 경로만 쓴다."""
    x = meae.pad(ds.tensor(np.array([i])).to(dev), pad)
    cut = lambda y: meae.crop(y, pad).squeeze().cpu().numpy().astype(np.float64)
    comps = [cut(model.component(x, k)) for k in range(K)]
    out = {'입력': noisy[i], '참값': clean[i],
           'M0 재구성': cut(model(x)[0]),
           'B 복원': cut(model.masked_reconstruct(x, k_noise)),
           '성분': comps}
    out['A 복원'] = noisy[i] - sum(comps[k] for k in k_noise)
    return out


def corr1(a, b):
    a = a - a.mean()
    b = b - b.mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float(abs((a * b).sum() / d)) if d > 0 else np.nan


def find(name=None, record=None, snr=None, n=10):
    """분절 목록. name 정확 일치 · record 기록번호 · snr (lo, hi) dB."""
    m = np.ones(len(ds), bool)
    if name:
        m &= np.array([x == name for x in names])
    if record:
        m &= np.array([d['record_id'] == str(record) for d in ds.meta])
    if snr:
        m &= (snr_in >= snr[0]) & (snr_in < snr[1])
    idx = np.where(m)[0][:n]
    return pd.DataFrame({'i': idx, '분절': [names[i] for i in idx],
                         '입력SNR': snr_in[idx].round(2),
                         'bw': [ds.meta[i]['snr_bw'] for i in idx],
                         'ma': [ds.meta[i]['snr_ma'] for i in idx],
                         'em': [ds.meta[i]['snr_em'] for i in idx]}).round(2)


find(snr=(-99, -2), n=8)


In [ ]:
# 복원 품질 순위 — 05 와 같은 기준(B ↔ x_clean 의 |r|)이다.
@torch.no_grad()
def rank_all(batch=100):
    r = np.zeros(len(ds))
    for s in range(0, len(ds), batch):
        j = np.arange(s, min(s + batch, len(ds)))
        x = meae.pad(ds.tensor(j).to(dev), pad)
        b = meae.crop(model.masked_reconstruct(x, k_noise), pad).squeeze(1)
        b = b.cpu().numpy().astype(np.float64)
        c = clean[j]
        a0 = c - c.mean(-1, keepdims=True)
        b0 = b - b.mean(-1, keepdims=True)
        d = np.sqrt((a0 ** 2).sum(-1) * (b0 ** 2).sum(-1))
        r[j] = np.abs((a0 * b0).sum(-1) / np.maximum(d, 1e-30))
    return r


rank = rank_all()
order = np.argsort(rank)
mid = len(order) // 2
picks = {'상위': order[-4:][::-1], '중간': order[mid - 2:mid + 2], '하위': order[:4]}
pd.DataFrame([{'구간': g, 'i': int(i), '분절': names[i],
               'B corr': round(rank[i], 4), '입력SNR': round(snr_in[i], 2)}
              for g, v in picks.items() for i in v])


## 3. 그림 — 고른 분절 하나

`SEG` 만 바꾸면 아래 그림이 전부 그 분절로 다시 그려진다.


In [ ]:
# ---- 여기만 바꾼다 -------------------------------------------------------
SEG = names[int(picks['중간'][0])]        # 예: '101_0051' 또는 names[0]
# -------------------------------------------------------------------------

i = names.index(SEG)
R = infer(i)
t = np.arange(len(clean[i])) / fs
m = ds.meta[i]
head = ('{} · {} · 분절 {} — 입력 SNR {:.2f} dB '
        '(주입 bw {:.1f} / ma {:.1f} / em {:.1f} dB)').format(
    RUN, SPLIT, SEG, snr_in[i], m['snr_bw'], m['snr_ma'], m['snr_em'])
print(head)


### 3.1 성분 적층 — 입력 · 재구성 · 성분 K개 · 참조 4종


In [ ]:
rows = ([('입력 x_noisy', R['입력'], '#000'),
         ('재구성 x_hat', R['M0 재구성'], '#d62728')]
        + [('성분 {}'.format(k + 1), R['성분'][k], '#1f77b4') for k in range(K)]
        + [('참조 {}'.format(r), ds.refs[r][i],
            '#2ca02c' if r == 'x_clean' else '#ff7f0e') for r in sup])
fig, ax = plt.subplots(len(rows), 1, figsize=(12, 0.95 * len(rows)), sharex=True)
for a, (lab, v, c) in zip(ax, rows):
    a.plot(t, v, lw=0.6, color=c)
    a.set_title(lab, fontsize=8, loc='left')
    a.grid(alpha=.25, lw=.4)
    a.tick_params(labelsize=7)
ax[-1].set_xlabel('시간 (초)')
fig.suptitle(head, fontsize=11)
fig.tight_layout()
plt.show()


### 3.2 배정 쌍 겹침 — 성분 k 와 배정 참조를 한 축에

쌍 안에서 y 범위를 공유한다. 쌍끼리 다른 것은 참조의 진폭이 다르기 때문이다.


In [ ]:
fig, ax = plt.subplots(K, 1, figsize=(12, 1.5 * K), sharex=True)
for k in range(K):
    a, ref = ax[k], ds.refs[sup[k]][i]
    a.plot(t, ref, lw=1.0, color='#ff7f0e', alpha=.75, label='참조 ' + sup[k])
    a.plot(t, R['성분'][k], lw=0.7, color='#1f77b4', label='성분 {}'.format(k + 1))
    lim = max(np.abs(ref).max(), np.abs(R['성분'][k]).max()) * 1.1
    a.set_ylim(-lim, lim)
    a.set_title('성분 {} ↔ {}   |r| {:.3f}   y ±{:.2f} mV'.format(
        k + 1, sup[k], corr1(ref, R['성분'][k]), lim), fontsize=8, loc='left')
    a.legend(fontsize=6.5, ncol=2, loc='upper right')
    a.grid(alpha=.25, lw=.4)
    a.tick_params(labelsize=7)
ax[-1].set_xlabel('시간 (초)')
fig.suptitle(head, fontsize=11)
fig.tight_layout()
plt.show()


### 3.3 처리 전 / 처리 후 겹침 — 05 와 같은 형식

네 칸 모두 같은 y 범위다. 잔차를 함께 두어 무엇이 남았는지 본다.


In [ ]:
panes = [('처리 전 — x_clean 과 x_noisy', R['입력'], '#000', 'x_noisy'),
         ('처리 후 — x_clean 과 B 복원', R['B 복원'], '#c44e52', 'B 복원')]
lim = max(np.abs(R['참값']).max(),
          *[np.abs(v).max() for _, v, _, _ in panes]) * 1.08
fig, ax = plt.subplots(4, 1, figsize=(12, 9.5), sharex=True, sharey=True)
for p, (title, v, c, lab) in enumerate(panes):
    a0, a1 = ax[2 * p], ax[2 * p + 1]
    a0.plot(t, v, lw=0.75, color=c, alpha=.8, label=lab)
    a0.plot(t, R['참값'], lw=0.95, color='#1f77b4', label='x_clean')
    a0.legend(fontsize=8, ncol=2, loc='upper right')
    a0.set_title('{}   |r| {:.3f} · RMSE {:.3f} mV'.format(
        title, corr1(R['참값'], v),
        float(np.sqrt(((R['참값'] - v) ** 2).mean()))), fontsize=9, loc='left')
    a1.plot(t, R['참값'] - v, lw=0.7, color='#777')
    a1.set_title('잔차 x_clean - ' + lab, fontsize=9, loc='left')
for a in ax:
    a.set_ylim(-lim, lim)
    a.set_ylabel('mV', fontsize=8)
    a.grid(alpha=.28, lw=.4)
    a.tick_params(labelsize=7)
ax[-1].set_xlabel('시간 (초)')
fig.suptitle(head + '   ·   전 칸 같은 y 범위 (±{:.2f} mV)'.format(lim), fontsize=11)
fig.tight_layout()
plt.show()


### 3.4 복원 방식 비교 — B · A · 고전 3종 · 기준선

04 와 같은 계산이다 (`metrics.classical_denoise`).


In [ ]:
cls = metrics.classical_denoise(R['입력'][None, :], fs)
cand = {'x_noisy (기준선)': R['입력'], 'B 복원': R['B 복원'], 'A 성분차감': R['A 복원']}
cand.update({k: v[0] for k, v in cls.items()})

ref2 = R['참값'][None, :]
s0 = float(metrics.snr_db_vec(ref2, R['입력'][None, :])[0])
rows = []
for k, v in cand.items():
    s = float(metrics.snr_db_vec(ref2, v[None, :])[0])
    rows.append({'방식': k, 'SNR_dB': s, 'dSNR': s - s0,
                 'PRD': float(metrics.prd(R['참값'], v)),
                 'corr': corr1(R['참값'], v),
                 'RMSE': float(np.sqrt(((R['참값'] - v) ** 2).mean()))})
display(pd.DataFrame(rows).round(4))

lim = max(np.abs(v).max() for v in cand.values()) * 1.08
fig, ax = plt.subplots(len(cand), 1, figsize=(12, 1.5 * len(cand)),
                       sharex=True, sharey=True)
for a, (k, v) in zip(ax, cand.items()):
    a.plot(t, R['참값'], lw=0.9, color='#1f77b4', alpha=.7, label='x_clean')
    a.plot(t, v, lw=0.7, color='#c44e52', label=k)
    a.set_ylim(-lim, lim)
    a.legend(fontsize=6.5, ncol=2, loc='upper right')
    a.set_title('{}   |r| {:.3f}'.format(k, corr1(R['참값'], v)), fontsize=8, loc='left')
    a.grid(alpha=.25, lw=.4)
    a.tick_params(labelsize=7)
ax[-1].set_xlabel('시간 (초)')
fig.suptitle(head, fontsize=11)
fig.tight_layout()
plt.show()


### 3.5 SQI — 그 분절 하나의 품질 지수 (참값 불필요)


In [ ]:
pk = metrics.detect_rpeaks(R['참값'], fs)      # 기준점을 세 계열에 공유한다
rows = []
for lab, v in (('x_clean', R['참값']), ('x_noisy', R['입력']), ('B 복원', R['B 복원'])):
    q = metrics.sqi_all(v[None, :], fs)
    row = {'계열': lab}
    row.update({k: float(x[0]) for k, x in q.items()})
    row['ECGMeanCoef'] = metrics.ecg_mean_coef(v, fs, peaks=pk)
    rows.append(row)
pd.DataFrame(rows).round(4)


## 4. 성능 지표 — 직접 계산

split 전체를 다시 계산한다. 저장된 CSV 를 그냥 읽으려면 §5 로 간다.


### 4.1 03 — 성분 정렬 (지표 6종)


In [ ]:
idx = np.arange(len(ds))
comps, refs = component_bank(model, ds, dev, idx)

cm = np.abs(pearson(comps, refs))
rbar, rsd, _ = aggregate(cm)
rn = rmse_norm_matrix(comps, refs).mean(0)
mdv = mad_matrix(comps, refs).mean(0)
ix, refs_c = enc_names(K), list(REF_KEYS)

col_clean = refs_c.index('x_clean')
pk_ref = metrics.reference_peaks(refs[:, col_clean], fs, progress=500)
f1 = metrics.f1_vector(comps, refs[:, col_clean], fs, pk_ref, progress=500)
rq = metrics.r_qrs_matrix(comps, refs, fs)

diag = []
for k, key in enumerate(sup):
    j = refs_c.index(key)
    off = [t for t in range(len(refs_c)) if t != j]
    jo = off[int(np.argmax(rbar[k, off]))]
    diag.append({'인코더': ix[k], '배정참조': key,
                 'corr': rbar[k, j], 'rmse_norm': rn[k, j], 'mad': mdv[k, j],
                 'F1': np.nanmean(f1[:, k]),
                 '누출_최대참조': refs_c[jo],
                 '누출비': rbar[k, jo] / max(rbar[k, j], 1e-12),
                 'r_QRS': rq[:, k, j].mean() if key == 'x_clean' else np.nan})
pd.DataFrame(diag).round(4)


### 4.2 04 — 복원 (고전 비교선 포함)


In [ ]:
tab = S04.three_ways(run=ckpt, split=SPLIT, outdir='_scratch/04')
tab


### 4.3 04 — 입력 SNR 구간별 분해


In [ ]:
br = pd.read_csv('_scratch/04/breakdown.csv', encoding='utf-8-sig')
sn = br[br['구간'].str.startswith('SNR')]
for col in ('dSNR_vs_입력', 'PRD'):
    print('[' + col + ']')
    display(sn.pivot(index='방식', columns='구간', values=col).round(3))


### 4.4 06 — SQI 와 임상 형태 지표

bSQI 가 분절마다 검출을 두 번 하므로 몇 분 걸린다.


In [ ]:
beats, summ, err_rec, sqi = S06.main(run=ckpt, split=SPLIT, outdir='_scratch/06')
display(sqi[['계열', '분절수'] + list(S06.SQI_KEYS)].round(4))
display(summ[['지표', '박동수', '참값_중앙', '처리전_절대오차중앙',
              '처리후_절대오차중앙', '처리전_편향중앙', '처리후_편향중앙',
              '개선된_박동비율']].round(4))


## 5. 저장된 산출물 읽기 — 재계산 없이


In [ ]:
def read(stage, fname, run=RUN, split=SPLIT, root='results'):
    return pd.read_csv(os.path.join(root, stage, run, split, fname),
                       encoding='utf-8-sig')


print('03 배정 대각')
display(read('03_bss', 'assignment_diagonal.csv').round(4))
print('04 복원')
display(read('04_masked_denoising', 'three_ways.csv').round(4))
print('06 SQI')
display(read('06_ablation', 'sqi_summary.csv')[
    ['계열', '분절수'] + list(S06.SQI_KEYS)].round(4))
print('06 임상 형태 지표')
display(read('06_ablation', 'metric_summary.csv').round(4))
print('05 그림 분절')
display(read('05_figure', 'segments.csv').round(4))


In [ ]:
# 구간별 분해 — 04 · 06 양쪽
for stage in ('04_masked_denoising', '06_ablation'):
    b = read(stage, 'breakdown.csv')
    sn = b[b['구간'].str.startswith('SNR')]
    key = '방식' if stage.startswith('04') else '지표'
    val = 'dSNR_vs_입력' if stage.startswith('04') else '개선된_박동비율'
    print('[{}] {}'.format(stage, val))
    display(sn.pivot(index=key, columns='구간', values=val).round(3))
